# Raw repertoires to annotation tables

Offline, tiny synthetic cohort and wide paired inputs. Repeated source IDs in different samples and a missing beta chain remain visible. Download the complete examples bundle so the shell script and raw inputs are beside this notebook.

In [ ]:
# Identify the installed CLI and print the fresh kernel environment.
import os, sys, subprocess, tempfile, json
from pathlib import Path
from importlib.metadata import version
os.environ["POLARS_MAX_THREADS"] = "1"
print("Python:", sys.version, "CLI interpreter:", sys.executable)
for package in ("vdjmatch", "vdjtools", "seqtree", "polars"):
    print(package, version(package))
# Open this notebook from the repository root or the examples directory.
examples = next((p for p in (Path.cwd()/"examples", Path.cwd())
                 if (p/"inputs/reference.tsv").is_file()), None)
if examples is None:
    raise RuntimeError("Use the downloaded complete examples bundle, not this notebook alone")
# Deterministic demonstration: no sampling or RNG seeds are used.
output = Path(tempfile.mkdtemp(prefix="vdjmatch-example-"))/"fresh"
output.mkdir()
print("Fresh output directory:", output)


In [ ]:
# Run raw inputs with this kernel's installed CLI, on any supported platform.
subprocess.run([sys.executable, "-m", "vdjmatch.cli", "annotate",
                "--metadata", str(examples/"inputs/samples.tsv"),
                "--vdjdb", str(examples/"inputs/reference.tsv"), "--threads", "4",
                "--output-prefix", str(output/"cohort")], check=True, timeout=120)
subprocess.run([sys.executable, "-m", "vdjmatch.cli", "annotate",
                str(examples/"inputs/paired.tsv"), "--locus", "paired",
                "--vdjdb", str(examples/"inputs/reference.tsv"), "--threads", "4",
                "--output-prefix", str(output/"paired")], check=True, timeout=120)

In [ ]:
# Analyze only final CLI tables, preserving sample and source identities.
import polars as pl
for prefix in ("cohort", "paired"):
    queries = pl.read_csv(output/(prefix+".queries.tsv"), separator="\t", infer_schema_length=0)
    selections = pl.read_csv(output/(prefix+".selection.tsv"), separator="\t", infer_schema_length=0)
    print(prefix, "query rows:", queries.height, "selection rows:", selections.height)
    identity = [c for c in ("sample_id", "source_sequence_id", "clone_id", "status", "has_support")
                if c in selections.columns]
    print(selections.select(identity))
    manifest = json.loads((output/(prefix+".manifest.json")).read_text())
    print("Native threads:", manifest["native_threads"])
assert pl.read_csv(output/"cohort.queries.tsv", separator="\t").height == 4
assert pl.read_csv(output/"paired.queries.tsv", separator="\t").height == 2


All displayed results come from fresh CLI tables. Raw synthetic `DEMO_A`/`DEMO_B` labels are software examples, not experimental binding truth. Keep no-hits and missing-chain dispositions. This example establishes neither accuracy nor a false-call threshold. Inspect the CLI manifests beside the tables for model, reference and resource settings. No computed outputs are committed or reused.